In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
import sys
from tqdm.auto import tqdm
from antropy import perm_entropy

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, lowpass_filter_causal, bootstrap_report

In [ ]:
sampling_rate = 100 # Hz, sensor sampling frequency
lpf_cutoff = 10 # Hz, FIR lowpass cutoff
lpf_order = 64 # FIR filter taps
pe_order = 5 # permutation entropy embedding dimension
pe_delay = int(sampling_rate / (2 * lpf_cutoff)) # PE delay (Nyquist-matched)
alpha = 0.023 # false-alarm bound
glue = False # set True to glue metrics for result collection
per_ride_path = None # set a path to save per-ride outcomes as CSV

In [ ]:
threshold = 1 - alpha # non-sober probability decision threshold

In [ ]:
data = pd.read_parquet("../data/data.parquet")
data

In [ ]:
labels = data.groupby(["ride_id", "participant_id"])["run"].first().reset_index()
labels

In [ ]:
features = ["ax", "ay", "az", "rx", "ry", "rz", "throttle"]

data_np = reshape_to_numpy(
    data,
    features = features,
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_lpf = lowpass_filter_causal(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=lpf_cutoff,
    gyro_cutoff=lpf_cutoff,
    order=lpf_order,
    fs=sampling_rate,
)

In [ ]:
# Compute one permutation-entropy value per channel per ride.
rows = []
for ride in tqdm(data_np_lpf, desc="entropy"):
    ent = []
    for ch in ride.T:
        ch = ch[~np.isnan(ch)] # strip FIR warm-up and trailing NaNs
        pe = perm_entropy(ch, order=pe_order, delay=pe_delay, normalize=True)
        ent.append(pe)
    rows.append(ent)

# convert to numpy
X_feat = np.array(rows)
X_feat.shape

In [ ]:
# Within-subject centering (WSC): subtract per-channel mean of the
# participant's other rides to remove between-subject variability
# without needing to know which ride is sober.
X_wsc = np.empty_like(X_feat)
for i in range(len(labels)): # i is a positional index, not a label
    # mask of the same participant's other rides
    pid = labels["participant_id"].iloc[i]
    others = (labels["participant_id"] == pid)
    others.iloc[i] = False
    # subtract their per-channel mean to center within subject
    X_wsc[i] = X_feat[i] - X_feat[others.values].mean(axis=0)

In [ ]:
# LOPO CV: fit a 3-class logistic regression per held-out participant
# and collect per-ride class probabilities.
logo = LeaveOneGroupOut()
groups = labels["participant_id"]
y_true_all, y_proba_all, pid_all = [], [], []

for train_idx, test_idx in tqdm(list(logo.split(X_wsc, labels["run"], groups)), desc="LOPO"):
    # fit on training set
    clf = make_pipeline(StandardScaler(), LogisticRegression())
    clf.fit(X_wsc[train_idx], labels["run"].iloc[train_idx])
    # infer on test set and append to results
    y_true_all.extend(labels["run"].iloc[test_idx])
    pid_all.extend(labels["participant_id"].iloc[test_idx])
    y_proba_all.append(clf.predict_proba(X_wsc[test_idx]))

# stack results
y_proba_all = np.vstack(y_proba_all)

In [ ]:
# detection rule: flag a ride as impaired if P(non-sober) >= threshold
y_true_arr = np.array(y_true_all) # truth to numpy
sober_col = np.where(clf.classes_ == "Sober")[0][0] # column index of the Sober class
p_non_sober = 1 - y_proba_all[:, sober_col] # non-sober probability per ride
detected = p_non_sober >= threshold # apply detection threshold
df_res = pd.DataFrame({"run": y_true_arr, "participant_id": pid_all, "detected": detected}) # make results 
df_res

In [ ]:
_ = bootstrap_report(df_res, glue=glue, per_ride_path=per_ride_path)